# Note
From 2009 - 2018 missing phase 1, 2A(1), 2A(2) applied

(Maybe for this period can use taken numbers instead of applied?)

In [2]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

YEARS = range(2009, 2026)
BASE = "https://sgschooling.com/year/{year}/all.html"
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def cell_value(text: str, kind: str) -> int:
    s = " ".join(text.split())
    if not s or s == "-":
        return 0

    # Prefer a standalone integer (e.g. "52 SC<1 74/52" -> 52)
    m = re.search(r"(?<!/)\b\d+\b(?!/)", s)
    if m:
        return int(m.group())

    # Fallback if cell is only a ratio (e.g. "74/52")
    r = re.search(r"(\d+)\s*/\s*(\d+)", s)
    if r:
        a, t = int(r.group(1)), int(r.group(2))
        return a if kind == "applied" else t

    return 0

def parse_year(year: int):
    url = BASE.format(year=year)
    html = requests.get(url, headers=HEADERS, timeout=40).text
    soup = BeautifulSoup(html, "html.parser")

    rows = soup.select("tr")
    out = []
    current_school = None
    current_applied = None

    for tr in rows:
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue

        texts = [" ".join(c.get_text(" ", strip=True).split()) for c in cells]
        first = texts[0] if texts else ""

        # school row (name row)
        if first and not first.startswith("↳") and first.lower() not in {"school"}:
            # likely school row: first cell has name, others mostly empty
            if any(k in first for k in ["P1 Ballot History", "All Primary Schools"]):
                continue
            if re.search(r"[A-Za-z]", first):
                current_school = first
                current_applied = None
            continue

        # applied row
        if first.startswith("↳ Applied") and current_school:
            vals = [cell_value(x, "applied") for x in texts[1:7]]
            current_applied = sum(vals)
            continue

        # taken row
        if first.startswith("↳ Taken") and current_school:
            vals = [cell_value(x, "taken") for x in texts[1:7]]
            total_taken = sum(vals)
            total_applied = current_applied if current_applied is not None else 0
            out.append({
                "school": current_school,
                "year": year,
                "total_applied": total_applied,
                "total_taken": total_taken,
            })
            current_applied = None

    return out

all_rows = []
for y in YEARS:
    yr = parse_year(y)
    print(y, len(yr))
    all_rows.extend(yr)

df = pd.DataFrame(all_rows, columns=["school", "year", "total_applied", "total_taken"])
df = (
    df.groupby(["school", "year"], as_index=False)[["total_applied", "total_taken"]]
      .max()
      .sort_values(["year", "school"])
      .reset_index(drop=True)
)


print(df.head(20))
print(df.shape)
print(df[["total_applied", "total_taken"]].describe())


2009 168
2010 169
2011 170
2012 177
2013 180
2014 180
2015 183
2016 183
2017 184
2018 184
2019 185
2020 186
2021 181
2022 181
2023 181
2024 180
2025 179
                     school  year  total_applied  total_taken
0                 Admiralty  2009            190          301
1             Ahmad Ibrahim  2009             85          146
2                   Ai Tong  2009            129          334
3              Anchor Green  2009            195          240
4                  Anderson  2009            141          240
5                Ang Mo Kio  2009            126          207
6    Anglo-Chinese (Junior)  2009            136          271
7   Anglo-Chinese (Primary)  2009             84          241
8                   Angsana  2009             30           61
9                    Beacon  2009            205          240
10              Bedok Green  2009            113          197
11                Bendemeer  2009             63          118
12             Blangah Rise  2009        

In [ ]:
# ---- build vacancy df ----
def parse_vacancy_year(year: int):
    url = BASE.format(year=year)
    html = requests.get(url, headers=HEADERS, timeout=40).text
    soup = BeautifulSoup(html, "html.parser")

    out = []
    current_school = None

    for tr in soup.select("tr"):
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue

        texts = [" ".join(c.get_text(" ", strip=True).split()) for c in cells]
        first = texts[0] if texts else ""

        # school row
        if first and not first.startswith("↳") and first.lower() not in {"school"}:
            if any(k in first for k in ["P1 Ballot History", "All Primary Schools"]):
                continue
            if re.search(r"[A-Za-z]", first):
                current_school = first
            continue

        # vacancy row
        if first.startswith("↳ Vacancy") and current_school:
            m = re.search(r"\((\d+)\)", first)
            vacancy = int(m.group(1)) if m else 0
            out.append({
                "school": current_school,
                "year": year,
                "vacancy": vacancy
            })

    return out

vacancy_rows = []
for y in YEARS:
    vacancy_rows.extend(parse_vacancy_year(y))

vacancy_df = pd.DataFrame(vacancy_rows, columns=["school", "year", "vacancy"])
vacancy_df = (
    vacancy_df.groupby(["school", "year"], as_index=False)["vacancy"]
    .max()
)

# ---- merge with your existing df ----
df_merged = df.merge(vacancy_df, on=["school", "year"], how="left")

# optional: fill missing vacancy with 0
df_merged["vacancy"] = df_merged["vacancy"].fillna(0).astype(int)

print(df_merged.head(20))
print(df_merged.shape)


                     school  year  total_applied  total_taken  vacancy
0                 Admiralty  2009            190          301      300
1             Ahmad Ibrahim  2009             85          146      180
2                   Ai Tong  2009            129          334      330
3              Anchor Green  2009            195          240      240
4                  Anderson  2009            141          240      240
5                Ang Mo Kio  2009            126          207      210
6    Anglo-Chinese (Junior)  2009            136          271      270
7   Anglo-Chinese (Primary)  2009             84          241      240
8                   Angsana  2009             30           61      180
9                    Beacon  2009            205          240      240
10              Bedok Green  2009            113          197      210
11                Bendemeer  2009             63          118      150
12             Blangah Rise  2009             34           71      210
13    

# add indicator if school ballot for that year

In [4]:
def phase_value(text: str, kind: str) -> int:
    s = " ".join(text.split())
    if not s or s == "-":
        return 0

    # if ratio exists, use it directly
    r = re.search(r"(\d+)\s*/\s*(\d+)", s)
    if r:
        a, t = int(r.group(1)), int(r.group(2))
        return a if kind == "applied" else t

    # otherwise use first standalone integer
    m = re.search(r"\b\d+\b", s)
    return int(m.group()) if m else 0

def parse_ballot_year(year: int):
    url = BASE.format(year=year)
    html = requests.get(url, headers=HEADERS, timeout=40).text
    soup = BeautifulSoup(html, "html.parser")

    out = []
    current_school = None
    applied_vals = None

    for tr in soup.select("tr"):
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue

        texts = [" ".join(c.get_text(' ', strip=True).split()) for c in cells]
        first = texts[0] if texts else ""

        # school row
        if first and not first.startswith("↳") and first.lower() not in {"school"}:
            if any(k in first for k in ["P1 Ballot History", "All Primary Schools"]):
                continue
            if re.search(r"[A-Za-z]", first):
                current_school = first
                applied_vals = None
            continue

        # applied row
        if first.startswith("↳ Applied") and current_school:
            applied_vals = [phase_value(x, "applied") for x in texts[1:7]]
            continue

        # taken row -> decide ballot
        if first.startswith("↳ Taken") and current_school and applied_vals is not None:
            taken_vals = [phase_value(x, "taken") for x in texts[1:7]]
            ballot = int(any(a > t for a, t in zip(applied_vals, taken_vals)))
            out.append({
                "school": current_school,
                "year": year,
                "ballot": ballot
            })
            applied_vals = None

    return out

# build ballot df
ballot_rows = []
for y in YEARS:
    ballot_rows.extend(parse_ballot_year(y))

ballot_df = pd.DataFrame(ballot_rows, columns=["school", "year", "ballot"])
ballot_df = ballot_df.groupby(["school", "year"], as_index=False)["ballot"].max()

# merge into your merged df
df_final = df_merged.merge(ballot_df, on=["school", "year"], how="left")
df_final["ballot"] = df_final["ballot"].fillna(0).astype(int)

df_final.to_csv("../data/primary_school_data_2009_2025.csv", index=False)
print(df_final.head(20))
print(df_final.shape)


                     school  year  total_applied  total_taken  vacancy  ballot
0                 Admiralty  2009            190          301      300       1
1             Ahmad Ibrahim  2009             85          146      180       1
2                   Ai Tong  2009            129          334      330       1
3              Anchor Green  2009            195          240      240       1
4                  Anderson  2009            141          240      240       1
5                Ang Mo Kio  2009            126          207      210       0
6    Anglo-Chinese (Junior)  2009            136          271      270       1
7   Anglo-Chinese (Primary)  2009             84          241      240       1
8                   Angsana  2009             30           61      180       0
9                    Beacon  2009            205          240      240       1
10              Bedok Green  2009            113          197      210       1
11                Bendemeer  2009             63    